# 03. Synthetic User Generation & Two-Way Marketplace Simulation

**Objective:**
To test our recommendation system's accuracy, we need to generate a large number of synthetic user profiles.

To ensure the data will be appropreate, we are going to generate data based on `upwork_cv_train.csv` dataset.

### The Generation Architecture

*   **1. Category-Based Feature Extraction:**
    We cluster the existing dataset into distinct, non-overlapping categories. For each category, we compute weighted frequency distributions for skills, geographic locations, developer levels, and client ratings. This ensures that synthetic profiles accurately reflect the true Upwork market landscape.

*   **2. Skill Vector Simulation (The 6-3-1 Rule):**
    Skills are not assigned purely at random. To simulate a realistic professional profile (core stack + secondary technologies), skills are sampled using a weighted approach:
    *   **6 skills** are randomly selected from the Top 10 most frequent skills in the category.
    *   **3 skills** from the Top 11-20 segment.
    *   **1 skill** from the remaining long-tail subset.

*   **3. Demographics & Market Attributes:**
    Attributes such as Country, Developer Level (Junior/Middle/Senior), and Success Score are populated using weighted random choices (`random.choices(..., weights=...)`) driven by the actual frequency dictionaries of the respective category.

*   **4. Level-Adjusted Pricing Strategy:**
    To simulate a realistic economic equilibrium, a freelancer's `Expected Hourly Rate` is calculated dynamically based on the category baseline and their specific developer level:
    *   *Junior:* 50% of the category average $\pm$ uniform noise.
    *   *Middle:* 100% of the category average $\pm$ uniform noise.
    *   *Senior:* 160% of the category average $\pm$ uniform noise.

*   **5. Semantic Profile Generation:**
    Transformer—freelancer profiles are constructed using a strict text template.
    *(e.g., "Profile: [Level] [Role]. Experience with [Skill 1], [Skill 2]. Based in [Country]. Expected rate: [Rate]")*.
    This guarantees that the generated embeddings will perfectly align with the latent space of the FAISS index during the recall stage.

In [3]:
import json
import faiss
import ast
import random
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

C:\Develop\Recommender-System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=

In [4]:
PROJECT_ROOT = Path.cwd().parent

# Paths to configs
CONFIG_PATH = PROJECT_ROOT / "configs" / "upwork-jobs.json"
INPUT_CSV_PATH = PROJECT_ROOT / "data" / "processed" / "upwork_cv_train.csv"

# Directory where we will store embeddings and FAISS index
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

# Define exact filenames for our ML models/vectors
EMBEDDINGS_PATH = ARTIFACTS_DIR / "vacancies_embeddings.joblib"
FAISS_INDEX_PATH = ARTIFACTS_DIR / "vector_index.faiss"

CATEGORIES_PATH = PROJECT_ROOT / "data" / "raw" / "categories.json"


# Loading config
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

with open(CATEGORIES_PATH, "r", encoding="utf-8") as f:
    categories = json.load(f)

MODEL_NAME = config.get("MODEL_NAME", "all-MiniLM-L6-v2")
PROFILES_NUM = config.get("PROFILES_NUM", 5000)
df = pd.read_csv(INPUT_CSV_PATH)

In [5]:
print("Loading Transformer model...")
model = SentenceTransformer(MODEL_NAME)

print("Loading FAISS index...")
index = faiss.read_index(str(FAISS_INDEX_PATH))

categories_similarity = {}

for category_name, category_desc in categories.items():
    print(f"Searching for category cluster: '{category_name}'")

    query_text = f"{category_name}: {category_desc}"
    # 1. Making a query vector
    query_vector = model.encode([query_text])

    # 2. Normalize the query vector
    faiss.normalize_L2(query_vector)

    # 3. Look up the nearest neighbors in the FAISS index
    k_neighbors = 200
    distances, indices = index.search(query_vector, k_neighbors)
    matched_indices = indices[0].tolist()
    categories_similarity[category_name] = matched_indices


Loading Transformer model...


Loading weights: 100%|██████████| 112/112 [00:00<00:00, 2544.29it/s]


Loading FAISS index...
Searching for category cluster: 'Web Development'
Searching for category cluster: 'Mobile Development'
Searching for category cluster: 'Frontend Development'
Searching for category cluster: 'Backend Development'
Searching for category cluster: 'Full Stack Development'
Searching for category cluster: 'WordPress Development'
Searching for category cluster: 'Game Development'
Searching for category cluster: 'Desktop Application'
Searching for category cluster: 'DevOps & Cloud'
Searching for category cluster: 'Cybersecurity'
Searching for category cluster: 'AI & Machine Learning'
Searching for category cluster: 'Data Science'
Searching for category cluster: 'Database Administration'
Searching for category cluster: 'QA & Testing'
Searching for category cluster: 'UI Design'
Searching for category cluster: 'UX Design'
Searching for category cluster: 'Graphic Design'
Searching for category cluster: 'Motion Graphics'
Searching for category cluster: '3D Design & Modeling'


In [7]:
def parse_skills(skill_str):
    if pd.isna(skill_str):
        return []
    try:
        return ast.literal_eval(skill_str)
    except (ValueError, SyntaxError):
        return str(skill_str).split()

skills_frequency = {}
countries_frequency = {}
levels_frequency = {}
rate_avg = {}


for category, indices in categories_similarity.items():
    matched_rows = df.iloc[indices]
    parsed_skills = matched_rows["skills"].apply(parse_skills)

    exploded_skills = parsed_skills.explode()
    skills_frequency[category] = exploded_skills.value_counts(normalize=True).to_dict()

    countries_frequency[category] = matched_rows["Client_Country"].value_counts(normalize=True).to_dict()

    if "EX_level_demand" in matched_rows.columns:
        levels_frequency[category] = matched_rows["EX_level_demand"].value_counts(normalize=True).to_dict()

    rate_avg[category] = matched_rows["hour_rate"].mean()


{'Web Development': {'web development': 0.17166979362101314, 'web design': 0.13696060037523453, 'html': 0.08724202626641651, 'wordpress': 0.08630393996247655, 'css': 0.06566604127579738, 'javascript': 0.06566604127579738, 'php': 0.0450281425891182, 'website': 0.039399624765478425, 'html5': 0.03470919324577861, 'landing page': 0.0300187617260788, 'graphic design': 0.024390243902439025, 'web application': 0.016885553470919325, 'responsive design': 0.009380863039399626, 'api integration': 0.009380863039399626, 'website redesign': 0.009380863039399626, 'api': 0.0075046904315197, 'wordpress development': 0.0075046904315197, 'godaddy': 0.005628517823639775, 'user flow': 0.005628517823639775, 'search engine optimization': 0.005628517823639775, 'react': 0.004690431519699813, 'css 3': 0.004690431519699813, 'java': 0.004690431519699813, 'database administration': 0.004690431519699813, 'mysql': 0.004690431519699813, 'webflow': 0.004690431519699813, 'shopify': 0.004690431519699813, 'database devel